# Zen - model server on Kaggle: Qwen 3.6 35B-A3B

Runs **llama-server** (llama.cpp's own Linux CUDA build) with **Qwen 3.6 35B-A3B** on Kaggle's two T4 GPUs, with
Qwen's own settings, and opens a public address - protected by a password - for the bridge on your PC.
(The earlier notebook with the Gemma models too is kept as `zen-multimodel-server.backup-2026-10-17.ipynb`.)

**Before running** (right-hand panel, *Session options*):
1. **Accelerator:** `GPU T4 x2` (the model is split over both automatically).
2. **Internet:** `On` (Kaggle asks for a phone-verified account).
3. **Persistence:** `No persistence` - a kept llama-server comes back broken, so it is downloaded fresh.
4. *Optional:* **Add-ons → Secrets** → add `LLAMA_API_KEY` with a password of your choice, so it stays the same every session.

Then **Run All**. It downloads llama-server (about a minute) and the model (about 21 GB: a few minutes).
The last cell prints the **address** and **password** for the bridge and keeps the server running -
keep this tab open (or use *Save Version → Save & Run All* to run it in the background for up to 12 hours).
Stop the last cell, or the session, to stop serving.

What it runs (measured 2026-10-17): 2 slots of 64k tokens each (one for the chat, one for Zen's helpers), about
58 words/s on an empty slot and 33 when a slot is nearly full; about 7 GB of the GPUs stay free.

In [ ]:
# --- Settings -------------------------------------------------------------------------------------
# Qwen 3.6 35B-A3B: 35B in all, about 3B used per word - fast on the T4s. Unsloth's K-quant (UD-Q4_K_S) is the fast
# one here (with Qwen's built-in MTP and the IQ4_XS file it ran 10.6 words/s; this file without MTP, 48+).
REPO = "unsloth/Qwen3.6-35B-A3B-GGUF"
MODEL_FILE = "Qwen3.6-35B-A3B-UD-Q4_K_S.gguf"
VISION_FILE = "mmproj-F16.gguf"   # lets the model read pictures
# Qwen 3.6's own sampling, from its model card: thinking mode, general tasks. The presence penalty keeps long
# thinking from repeating itself. Zen's analyst asks per request for Qwen's precise-coding set (temperature 0.6,
# no penalty: app/assistant/workers/contracts.py).
SAMPLING = ["--temp", "1.0", "--top-p", "0.95", "--top-k", "20", "--min-p", "0.0", "--presence-penalty", "1.5"]
# Two slots - one for the chat, one for Zen's helpers - of 64k tokens each: CONTEXT is both together
CONTEXT = 131072
LLAMA_BUILD = "b11474"   # llama.cpp's ready-made Linux CUDA 12.8 build (Kaggle has CUDA 12.8)
PORT = 8080
WORK = "/kaggle/working"
MODELS = "/tmp/models/qwen36"

import glob, json, os, re, secrets, shutil, subprocess, time, urllib.request

# Runs a command with its output shown here as it comes; on an error, stops with the last lines of
# output so the reason is visible.
def sh(cmd, quiet_progress=False):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail, steps = [], 0
    for line in proc.stdout:
        tail = (tail + [line])[-40:]
        if quiet_progress and "%]" in line:
            steps += 1
            if steps % 25:
                continue
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"{cmd[0]} failed:\n" + "".join(tail))

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

In [ ]:
# --- llama-server: llama.cpp's ready-made Linux CUDA 12.8 build (about a minute), fresh each session:
# a copy kept by Persistence comes back without its run permission and its library links
BIN = f"/tmp/llama-bin/{LLAMA_BUILD}"
if not glob.glob(f"{BIN}/**/llama-server", recursive=True):
    os.makedirs(BIN, exist_ok=True)
    base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}"
    for name in (f"llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz",          # llama-server and its libraries
                 f"cudart-llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz"):   # the CUDA runtime it was built with
        urllib.request.urlretrieve(f"{base}/{name}", f"/tmp/{name}")
        sh(["tar", "-xzf", f"/tmp/{name}", "-C", BIN])
SERVER = glob.glob(f"{BIN}/**/llama-server", recursive=True)[0]
# where its libraries are: every folder of the download holding one
ENV = {**os.environ, "LD_LIBRARY_PATH": ":".join(sorted({os.path.dirname(p) for p in glob.glob(f"{BIN}/**/*.so*", recursive=True)})
                                                 + [os.environ.get("LD_LIBRARY_PATH", "")])}
version = subprocess.run([SERVER, "--version"], capture_output=True, text=True, env=ENV)
said = version.stdout + version.stderr
print(SERVER, "|", next((line for line in said.splitlines() if "version" in line), said.strip()[-300:]))

In [ ]:
# --- The model and its vision part, from Hugging Face (about 21 GB) ----------------------------------
from huggingface_hub import hf_hub_download

paths = {name: hf_hub_download(REPO, name, local_dir=MODELS) for name in (MODEL_FILE, VISION_FILE)}
for name, path in paths.items():
    print(f"{os.path.getsize(path) / 1e9:5.2f} GB  {name}")

In [ ]:
# --- The password: your Kaggle secret LLAMA_API_KEY if set, else this session's (kept when re-run) ----
if "API_KEY" not in globals():
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("LLAMA_API_KEY")
    except Exception:
        API_KEY = secrets.token_urlsafe(24)

# a server from an earlier run of this cell still holds the port and the GPUs: stop it first
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(60)

# --- Start llama-server: Qwen's sampling, both GPUs, 8-bit conversation memory, 2 slots ---------------
args = [SERVER, "-m", paths[MODEL_FILE], "--alias", "local", *SAMPLING,
        "--mmproj", paths[VISION_FILE], "--image-min-tokens", "280", "--image-max-tokens", "560",
        "--fit", "on", "--fit-target", "256", "--fit-ctx", str(CONTEXT),
        "-c", str(CONTEXT), "-np", "2", "--no-kv-unified", "-fa", "on", "-ctk", "q8_0", "-ctv", "q8_0",
        "-b", "1024", "-ub", "512", "--reasoning", "auto", "--reasoning-format", "deepseek", "--slots",
        "--api-key", API_KEY, "--host", "127.0.0.1", "--port", str(PORT)]
log_path = f"{WORK}/llama-server.log"
server = subprocess.Popen(args, stdout=open(log_path, "w"), stderr=subprocess.STDOUT, env=ENV)
for _ in range(300):
    if server.poll() is not None:
        raise RuntimeError("llama-server stopped:\n" + open(log_path).read()[-3000:])
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as response:
            if json.load(response).get("status") == "ok":
                break
    except Exception:
        pass
    time.sleep(2)
print("llama-server is up | Qwen 3.6 35B-A3B | two GPUs, 8-bit memory |",
      subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().replace("\n", " / "))

In [ ]:
# --- A quick speed check, straight to the server -----------------------------------------------------
def ask(prompt, max_tokens=300):
    body = {"messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens,
            "chat_template_kwargs": {"enable_thinking": False}}
    request = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", json.dumps(body).encode(),
                                     {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
    with urllib.request.urlopen(request, timeout=600) as response:
        out = json.load(response)
    t = out["timings"]
    print(f"prompt {t['prompt_n']} tokens at {t['prompt_per_second']:.0f}/s | "
          f"answer {t['predicted_n']} tokens at {t['predicted_per_second']:.1f}/s")
    print(out["choices"][0]["message"]["content"][:400])

ask("In three sentences: what makes a sales forecast trustworthy?")

In [ ]:
# --- Where the model sits: memory per GPU, and RAM (the model must not spill into RAM) ---------------
print(subprocess.run("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv; free -g",
                     shell=True, capture_output=True, text=True).stdout)

In [ ]:
# --- The public address: a Cloudflare quick tunnel (no account needed) -------------------------------
CLOUDFLARED = "/tmp/cloudflared"
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                               CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)
tunnel = subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=open("/tmp/cloudflared.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(90):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/cloudflared.log").read())
    if found:
        URL = found.group(0)
        break
    time.sleep(1)
if URL is None:
    raise RuntimeError("No tunnel address:\n" + open("/tmp/cloudflared.log").read()[-2000:])
print("=" * 72)
print("For the bridge on your PC:")
print("  address :", URL)
print("  password:", API_KEY)
print("=" * 72)

In [ ]:
# --- Keep serving (stop this cell to stop) ------------------------------------------------------------
# Shows every few minutes that the server and the tunnel are still up; the address stays the same while
# this session runs.
while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as response:
            state = json.load(response).get("status")
    except Exception as exc:
        state = f"DOWN ({exc})"
    print(time.strftime("%H:%M:%S"), "| server:", state, "| tunnel:",
          "up" if tunnel.poll() is None else "STOPPED", "|", URL, flush=True)
    time.sleep(300)